# 5.4 Mathematical Models and Least-Squares Analysis

**Course:** Elementary Linear Algebra (Larson, 8e)  
**Chapter 5 --- Inner Product Spaces**

---

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '../..'))

In [ ]:
from linalg_core.matrix import Matrix
from linalg_core.ops import multiply, transpose
from linalg_core.inner import least_squares, dot, norm
from linalg_core import EPSILON

import numpy as np
import matplotlib.pyplot as plt

print("Setup complete.")

---

## 2. Motivation

Suppose you record hourly temperature readings at a weather station and obtain
the following six data points:

| Hour $t$ | 0 | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|---|
| Temp $y$ (\textdegree F) | 32.1 | 33.5 | 36.2 | 40.1 | 45.3 | 51.0 |

The data looks roughly parabolic --- temperature accelerates upward.  You suspect
a model of the form

$$y = a + bt + ct^2$$

Plugging each data point into this equation gives **six** equations in **three**
unknowns $(a, b, c)$.  Unless the data lies *exactly* on a parabola (unlikely
with noisy measurements), **no** values of $a, b, c$ satisfy all six equations
simultaneously.  The system is **inconsistent**.

So we cannot solve it exactly.  But we *can* ask:

> **What values of $a, b, c$ make the errors as small as possible?**

That is the **least-squares problem**, and the answer comes from the
**normal equations** --- a beautiful application of inner products and
orthogonal projection.

In [ ]:
t_data = [0.0, 1.0, 2.0, 3.0, 4.0, 5.0]
y_data = [32.1, 33.5, 36.2, 40.1, 45.3, 51.0]

plt.figure(figsize=(7, 4))
plt.scatter(t_data, y_data, color='crimson', s=80, zorder=5, label='Measured data')
plt.xlabel('Hour $t$')
plt.ylabel('Temperature ($^\circ$F)')
plt.title('Noisy Temperature Readings --- Which Curve Fits Best?')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---

## 3. Build --- Core Concepts

### 3.1 Inconsistent Systems --- More Equations Than Unknowns

An $m \times n$ system $A\mathbf{x} = \mathbf{b}$ with $m > n$ is
**overdetermined**: there are more constraints (equations) than free
parameters (unknowns).  Generically, such a system has **no** solution.

**Concrete example.**  Fit a line $y = a + bt$ through three non-collinear
points $(0, 1)$, $(1, 3)$, $(2, 2)$:

$$
\begin{cases}
a + 0 \cdot b = 1 \\
a + 1 \cdot b = 3 \\
a + 2 \cdot b = 2
\end{cases}
\quad\Longrightarrow\quad
\underbrace{\begin{bmatrix}1 & 0\\1 & 1\\1 & 2\end{bmatrix}}_{A}
\underbrace{\begin{bmatrix}a\\b\end{bmatrix}}_{\mathbf{x}}
=
\underbrace{\begin{bmatrix}1\\3\\2\end{bmatrix}}_{\mathbf{b}}
$$

From the first two equations, $a = 1$ and $b = 2$.  But then the third
equation gives $1 + 2(2) = 5 \neq 2$.  **No solution.**

We need a different strategy: instead of solving $A\mathbf{x} = \mathbf{b}$
exactly, find the $\hat{\mathbf{x}}$ that gets **as close as possible**.

In [ ]:
A_demo = Matrix.from_lists([[1, 0], [1, 1], [1, 2]])
b_demo = [1.0, 3.0, 2.0]

print("A ="); print(A_demo)
print(f"\nb = {b_demo}")
print(f"\nSystem: 3 equations, 2 unknowns -> overdetermined")
print(f"No exact solution exists (the three points are not collinear).")

### 3.2 The Least-Squares Idea --- Minimize $\|A\mathbf{x} - \mathbf{b}\|^2$

Since $A\mathbf{x} = \mathbf{b}$ has no solution, we define the
**residual vector**

$$\mathbf{r} = \mathbf{b} - A\hat{\mathbf{x}}$$

which measures how far each equation is from being satisfied.  The
**least-squares solution** $\hat{\mathbf{x}}$ minimizes the sum of
squared residuals:

$$\hat{\mathbf{x}} = \arg\min_{\mathbf{x}} \|A\mathbf{x} - \mathbf{b}\|^2
= \arg\min_{\mathbf{x}} \sum_{i=1}^{m}(a_i^T \mathbf{x} - b_i)^2$$

**Why squared?**
- Squaring penalizes large errors more heavily than small ones.
- The squared norm is differentiable and leads to a *linear* system for $\hat{\mathbf{x}}$.
- Geometrically, $A\hat{\mathbf{x}}$ is the point in $\text{Col}(A)$ closest
  to $\mathbf{b}$ --- this is an **orthogonal projection**.

The individual residual for equation $i$ is $r_i = b_i - (\text{row}_i \text{ of } A) \cdot \hat{\mathbf{x}}$.  Visually, $|r_i|$ is the vertical distance from data point $i$ to the fitted curve.

In [ ]:
x_try1 = [1.0, 2.0]
x_try2 = [1.5, 0.5]

def compute_residual(A, x_vec, b_vec):
    """Compute residual r = b - Ax and its squared norm."""
    x_mat = Matrix.from_vector(x_vec)
    Ax = multiply(A, x_mat)
    r = [b_vec[i] - Ax[i, 0] for i in range(len(b_vec))]
    sq_norm = sum(ri ** 2 for ri in r)
    return r, sq_norm

r1, sq1 = compute_residual(A_demo, x_try1, b_demo)
r2, sq2 = compute_residual(A_demo, x_try2, b_demo)

print("--- Trying x = [1, 2] (from first two equations) ---")
print(f"Residual r = {[f'{v:.2f}' for v in r1]}")
print(f"||r||^2 = {sq1:.4f}")

print("\n--- Trying x = [1.5, 0.5] (a guess) ---")
print(f"Residual r = {[f'{v:.2f}' for v in r2]}")
print(f"||r||^2 = {sq2:.4f}")

print(f"\nThe second guess has {'smaller' if sq2 < sq1 else 'larger'} total squared error.")
print("The least-squares solution will minimize this quantity.")

### 3.3 The Normal Equations --- $A^T A \hat{\mathbf{x}} = A^T \mathbf{b}$

> **Theorem (Larson, Sec. 5.4).** The least-squares solution
> $\hat{\mathbf{x}}$ of $A\mathbf{x} = \mathbf{b}$ satisfies the
> **normal equations**:
>
> $$A^T A\,\hat{\mathbf{x}} = A^T \mathbf{b}$$

**Derivation sketch.**  We want to minimize
$f(\mathbf{x}) = \|A\mathbf{x} - \mathbf{b}\|^2$.

Expand:
$$f(\mathbf{x}) = (A\mathbf{x} - \mathbf{b})^T(A\mathbf{x} - \mathbf{b})
= \mathbf{x}^T A^T A \mathbf{x} - 2\mathbf{x}^T A^T \mathbf{b}
+ \mathbf{b}^T \mathbf{b}$$

Set the gradient to zero:
$$\nabla f = 2 A^T A \mathbf{x} - 2 A^T \mathbf{b} = \mathbf{0}$$

$$\boxed{A^T A \,\hat{\mathbf{x}} = A^T \mathbf{b}}$$

**Geometric interpretation:** The residual $\mathbf{r} = \mathbf{b} - A\hat{\mathbf{x}}$
must be **orthogonal** to every column of $A$.  Writing this as
$A^T \mathbf{r} = \mathbf{0}$ and expanding gives the normal equations.

**When is $A^T A$ invertible?**  Exactly when the columns of $A$ are linearly
independent (which happens whenever $m \geq n$ and $\text{rank}(A) = n$).  In
practice this is almost always satisfied for regression problems.

In [ ]:
print("--- Forming the normal equations for the 3-point line fit ---")
print("\nA ="); print(A_demo)

AtA = multiply(transpose(A_demo), A_demo)
print("\nA^T A ="); print(AtA)

b_mat = Matrix.from_vector(b_demo)
Atb = multiply(transpose(A_demo), b_mat)
print("\nA^T b ="); print(Atb)

print("\nThe normal equations A^T A x = A^T b are:")
print(f"  [{AtA[0,0]:.0f}  {AtA[0,1]:.0f}] [a]   [{Atb[0,0]:.0f}]")
print(f"  [{AtA[1,0]:.0f}  {AtA[1,1]:.0f}] [b] = [{Atb[1,0]:.0f}]")

print("\nThis is a 2x2 system --- always solvable (A^T A is invertible here).")

### 3.4 Implement --- Demo `least_squares(A, b)`

Our library function `least_squares(A, b)` automates the normal equations:

1. Form $A^T A$ and $A^T \mathbf{b}$.
2. Factor $A^T A = LU$.
3. Solve $L\mathbf{y} = A^T \mathbf{b}$ by forward substitution.
4. Solve $U\hat{\mathbf{x}} = \mathbf{y}$ by back substitution.

Let us solve the 3-point example from Section 3.1.

In [ ]:
x_hat = least_squares(A_demo, b_demo)
print(f"Least-squares solution: a = {x_hat[0]:.6f}, b = {x_hat[1]:.6f}")
print(f"Best-fit line: y = {x_hat[0]:.4f} + {x_hat[1]:.4f} * t")

r_opt, sq_opt = compute_residual(A_demo, x_hat, b_demo)
print(f"\nResidual: r = {[f'{v:.4f}' for v in r_opt]}")
print(f"||r||^2 = {sq_opt:.6f}")

print(f"\nCompare with our earlier guesses:")
print(f"  x=[1, 2]     -> ||r||^2 = {sq1:.4f}")
print(f"  x=[1.5, 0.5] -> ||r||^2 = {sq2:.4f}")
print(f"  x_hat        -> ||r||^2 = {sq_opt:.6f}  (smallest!)")

### 3.5 Linear Regression --- Fit $y = a + bt$

Now we return to our temperature data and fit a **linear model**
$y = a + bt$.  Each data point $(t_i, y_i)$ gives the equation
$a + b t_i = y_i$.  Stacking all six equations:

$$
\underbrace{\begin{bmatrix}
1 & t_0 \\ 1 & t_1 \\ 1 & t_2 \\ 1 & t_3 \\ 1 & t_4 \\ 1 & t_5
\end{bmatrix}}_{\text{Vandermonde } V_{6 \times 2}}
\begin{bmatrix} a \\ b \end{bmatrix}
=
\begin{bmatrix} y_0 \\ y_1 \\ y_2 \\ y_3 \\ y_4 \\ y_5 \end{bmatrix}
$$

The matrix on the left is a **Vandermonde matrix** of degree 1: column $j$
contains $t_i^j$.  Solving via the normal equations gives the best-fit line.

In [ ]:
n_pts = len(t_data)

V_lin = Matrix(n_pts, 2)
for i in range(n_pts):
    V_lin[i, 0] = 1.0
    V_lin[i, 1] = t_data[i]

print("Vandermonde matrix V (degree 1):")
print(V_lin)

coeffs_lin = least_squares(V_lin, y_data)
a_lin, b_lin = coeffs_lin

print(f"\nLeast-squares coefficients:")
print(f"  a (intercept) = {a_lin:.6f}")
print(f"  b (slope)     = {b_lin:.6f}")
print(f"\nBest-fit line: y = {a_lin:.4f} + {b_lin:.4f} t")

y_pred_lin = [a_lin + b_lin * t for t in t_data]
resid_lin = [y_data[i] - y_pred_lin[i] for i in range(n_pts)]
ss_res_lin = sum(r ** 2 for r in resid_lin)
print(f"\nResiduals: {[f'{r:.4f}' for r in resid_lin]}")
print(f"Sum of squared residuals: {ss_res_lin:.6f}")

### 3.6 Quadratic Regression --- Fit $y = a + bt + ct^2$

A line does not capture the upward *curvature* we see in the data.
Fitting a **quadratic** model $y = a + bt + ct^2$ uses a $6 \times 3$
Vandermonde matrix:

$$
V = \begin{bmatrix}
1 & t_0 & t_0^2 \\
1 & t_1 & t_1^2 \\
\vdots & \vdots & \vdots \\
1 & t_5 & t_5^2
\end{bmatrix}
$$

The normal equations $V^T V \hat{\mathbf{x}} = V^T \mathbf{y}$ are now
$3 \times 3$, and the solution gives three coefficients $(a, b, c)$.

In [ ]:
V_quad = Matrix(n_pts, 3)
for i in range(n_pts):
    V_quad[i, 0] = 1.0
    V_quad[i, 1] = t_data[i]
    V_quad[i, 2] = t_data[i] ** 2

print("Vandermonde matrix V (degree 2):")
print(V_quad)

coeffs_quad = least_squares(V_quad, y_data)
a_q, b_q, c_q = coeffs_quad

print(f"\nLeast-squares coefficients:")
print(f"  a (constant)  = {a_q:.6f}")
print(f"  b (linear)    = {b_q:.6f}")
print(f"  c (quadratic) = {c_q:.6f}")
print(f"\nBest-fit parabola: y = {a_q:.4f} + {b_q:.4f} t + {c_q:.4f} t^2")

y_pred_quad = [a_q + b_q * t + c_q * t**2 for t in t_data]
resid_quad = [y_data[i] - y_pred_quad[i] for i in range(n_pts)]
ss_res_quad = sum(r ** 2 for r in resid_quad)
print(f"\nResiduals: {[f'{r:.4f}' for r in resid_quad]}")
print(f"Sum of squared residuals: {ss_res_quad:.6f}")

print(f"\n--- Comparison ---")
print(f"Linear  ||r||^2 = {ss_res_lin:.6f}")
print(f"Quadratic ||r||^2 = {ss_res_quad:.6f}")
print(f"Quadratic reduces error by {(1 - ss_res_quad/ss_res_lin)*100:.1f}%")

### 3.7 Projection Interpretation --- $A\hat{\mathbf{x}}$ Projects $\mathbf{b}$ onto $\text{Col}(A)$

The least-squares solution has a beautiful geometric meaning:

$$A\hat{\mathbf{x}} = \text{proj}_{\text{Col}(A)} \mathbf{b}$$

$A\hat{\mathbf{x}}$ is the **orthogonal projection** of $\mathbf{b}$ onto the
column space of $A$.  It is the point in $\text{Col}(A)$ that is **closest**
to $\mathbf{b}$.

The residual $\mathbf{r} = \mathbf{b} - A\hat{\mathbf{x}}$ is **orthogonal**
to every column of $A$:

$$A^T \mathbf{r} = A^T(\mathbf{b} - A\hat{\mathbf{x}}) = A^T \mathbf{b} - A^T A \hat{\mathbf{x}} = \mathbf{0}$$

This is precisely the normal equation rearranged.  The residual lives in the
**left null space** of $A$ (orthogonal complement of the column space).

Let us verify this orthogonality numerically for both our linear and quadratic
fits.

In [ ]:
print("=" * 60)
print("PROJECTION INTERPRETATION --- Residual orthogonal to Col(A)")
print("=" * 60)

print("\n--- Linear fit (6x2 Vandermonde) ---")
print(f"Residual r = {[f'{v:.6f}' for v in resid_lin]}")
for j in range(V_lin.cols):
    col_j = V_lin.get_col(j)
    dp = dot(resid_lin, col_j)
    print(f"  dot(r, col_{j}) = {dp:.2e}  {'~ 0' if abs(dp) < 1e-6 else 'NOT zero!'}")

At_r_lin = multiply(transpose(V_lin), Matrix.from_vector(resid_lin))
print(f"  A^T r = {[f'{At_r_lin[i,0]:.2e}' for i in range(At_r_lin.rows)]}")

print("\n--- Quadratic fit (6x3 Vandermonde) ---")
print(f"Residual r = {[f'{v:.6f}' for v in resid_quad]}")
for j in range(V_quad.cols):
    col_j = V_quad.get_col(j)
    dp = dot(resid_quad, col_j)
    print(f"  dot(r, col_{j}) = {dp:.2e}  {'~ 0' if abs(dp) < 1e-6 else 'NOT zero!'}")

At_r_quad = multiply(transpose(V_quad), Matrix.from_vector(resid_quad))
print(f"  A^T r = {[f'{At_r_quad[i,0]:.2e}' for i in range(At_r_quad.rows)]}")

print("\nIn both cases, A^T r is approximately zero,")
print("confirming the residual is orthogonal to the column space.")

---

## 4. Verify --- Cross-Check with NumPy

We compare our from-scratch results against NumPy's `np.linalg.lstsq` and
`np.polyfit` to ensure correctness.

In [ ]:
def to_np(mat):
    """Convert our Matrix to a NumPy array."""
    return np.array(mat.to_lists())

In [ ]:
print("=" * 60)
print("VERIFICATION: least_squares vs np.linalg.lstsq")
print("=" * 60)

t_np = np.array(t_data)
y_np = np.array(y_data)

print("\n--- Linear fit ---")
np_lin, np_res_lin, _, _ = np.linalg.lstsq(to_np(V_lin), y_np, rcond=None)
print(f"  Our coefficients:   a={a_lin:.8f}, b={b_lin:.8f}")
print(f"  NumPy coefficients: a={np_lin[0]:.8f}, b={np_lin[1]:.8f}")
lin_match = all(abs(coeffs_lin[i] - np_lin[i]) < 1e-6 for i in range(2))
print(f"  Match: {'PASS' if lin_match else 'FAIL'}")

print("\n--- Quadratic fit ---")
np_quad, np_res_quad, _, _ = np.linalg.lstsq(to_np(V_quad), y_np, rcond=None)
print(f"  Our coefficients:   a={a_q:.8f}, b={b_q:.8f}, c={c_q:.8f}")
print(f"  NumPy coefficients: a={np_quad[0]:.8f}, b={np_quad[1]:.8f}, c={np_quad[2]:.8f}")
quad_match = all(abs(coeffs_quad[i] - np_quad[i]) < 1e-6 for i in range(3))
print(f"  Match: {'PASS' if quad_match else 'FAIL'}")

In [ ]:
print("=" * 60)
print("VERIFICATION: coefficients vs np.polyfit")
print("=" * 60)

print("\n--- Linear fit (degree 1) ---")
pf_lin = np.polyfit(t_np, y_np, 1)
print(f"  Our:     a={a_lin:.8f}, b={b_lin:.8f}")
print(f"  polyfit: a={pf_lin[1]:.8f}, b={pf_lin[0]:.8f}  (note: polyfit returns highest degree first)")
pf_lin_match = abs(a_lin - pf_lin[1]) < 1e-6 and abs(b_lin - pf_lin[0]) < 1e-6
print(f"  Match: {'PASS' if pf_lin_match else 'FAIL'}")

print("\n--- Quadratic fit (degree 2) ---")
pf_quad = np.polyfit(t_np, y_np, 2)
print(f"  Our:     a={a_q:.8f}, b={b_q:.8f}, c={c_q:.8f}")
print(f"  polyfit: a={pf_quad[2]:.8f}, b={pf_quad[1]:.8f}, c={pf_quad[0]:.8f}")
pf_quad_match = (
    abs(a_q - pf_quad[2]) < 1e-6
    and abs(b_q - pf_quad[1]) < 1e-6
    and abs(c_q - pf_quad[0]) < 1e-6
)
print(f"  Match: {'PASS' if pf_quad_match else 'FAIL'}")

In [ ]:
print("=" * 60)
print("VERIFICATION: Residual orthogonality (A^T r ~ 0)")
print("=" * 60)

all_pass = True

test_cases = [
    ("3-point line", A_demo, b_demo),
    ("Temperature linear", V_lin, y_data),
    ("Temperature quadratic", V_quad, y_data),
]

for label, A_test, b_test in test_cases:
    x_test = least_squares(A_test, b_test)
    Ax_mat = multiply(A_test, Matrix.from_vector(x_test))
    r_test = [b_test[i] - Ax_mat[i, 0] for i in range(len(b_test))]
    Atr = multiply(transpose(A_test), Matrix.from_vector(r_test))
    max_entry = max(abs(Atr[i, 0]) for i in range(Atr.rows))
    passed = max_entry < 1e-6
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] {label}: max|A^T r| = {max_entry:.2e}")
    if not passed:
        all_pass = False

print(f"\nOverall: {'ALL PASSED' if all_pass else 'SOME FAILED'}")

---

## 5. Visualize

### 5.1 Scatter + Linear Fit + Quadratic Fit + Cubic Fit

In [ ]:
V_cub = Matrix(n_pts, 4)
for i in range(n_pts):
    V_cub[i, 0] = 1.0
    V_cub[i, 1] = t_data[i]
    V_cub[i, 2] = t_data[i] ** 2
    V_cub[i, 3] = t_data[i] ** 3

coeffs_cub = least_squares(V_cub, y_data)
a_c, b_c, c_c, d_c = coeffs_cub
print(f"Cubic fit: y = {a_c:.4f} + {b_c:.4f}t + {c_c:.4f}t^2 + {d_c:.4f}t^3")

y_pred_cub = [a_c + b_c*t + c_c*t**2 + d_c*t**3 for t in t_data]
resid_cub = [y_data[i] - y_pred_cub[i] for i in range(n_pts)]
ss_res_cub = sum(r**2 for r in resid_cub)

ss_tot = sum((y - np.mean(y_data))**2 for y in y_data)
R2_lin = 1 - ss_res_lin / ss_tot
R2_quad = 1 - ss_res_quad / ss_tot
R2_cub = 1 - ss_res_cub / ss_tot

print(f"\n--- Goodness of fit (R^2) ---")
print(f"  Linear:    R^2 = {R2_lin:.6f}")
print(f"  Quadratic: R^2 = {R2_quad:.6f}")
print(f"  Cubic:     R^2 = {R2_cub:.6f}")

In [ ]:
t_smooth = np.linspace(-0.5, 5.5, 200)

y_lin_smooth = a_lin + b_lin * t_smooth
y_quad_smooth = a_q + b_q * t_smooth + c_q * t_smooth**2
y_cub_smooth = a_c + b_c * t_smooth + c_c * t_smooth**2 + d_c * t_smooth**3

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(t_data, y_data, color='black', s=90, zorder=5, label='Data', edgecolors='white', linewidths=0.8)
ax.plot(t_smooth, y_lin_smooth, '--', color='steelblue', linewidth=2,
        label=f'Linear ($R^2 = {R2_lin:.4f}$)')
ax.plot(t_smooth, y_quad_smooth, '-', color='darkorange', linewidth=2,
        label=f'Quadratic ($R^2 = {R2_quad:.4f}$)')
ax.plot(t_smooth, y_cub_smooth, ':', color='seagreen', linewidth=2,
        label=f'Cubic ($R^2 = {R2_cub:.4f}$)')
ax.set_xlabel('Hour $t$', fontsize=12)
ax.set_ylabel('Temperature ($^\circ$F)', fontsize=12)
ax.set_title('Least-Squares Polynomial Fits', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

### 5.2 Residual Bars

Visualizing residuals reveals the *pattern* of errors.  Ideally, residuals
should look random (no systematic trend).  A clear pattern suggests the model
is missing structure in the data.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)

colors = ['steelblue', 'darkorange', 'seagreen']
labels = ['Linear', 'Quadratic', 'Cubic']
residuals = [resid_lin, resid_quad, resid_cub]
ss_vals = [ss_res_lin, ss_res_quad, ss_res_cub]

for ax, resid, color, label, ss in zip(axes, residuals, colors, labels, ss_vals):
    ax.bar(t_data, resid, width=0.4, color=color, alpha=0.7, edgecolor='black', linewidth=0.5)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_xlabel('Hour $t$')
    ax.set_title(f'{label}\n$\\Sigma r_i^2 = {ss:.4f}$')
    ax.grid(True, alpha=0.3, axis='y')

axes[0].set_ylabel('Residual $r_i$')
fig.suptitle('Residual Analysis by Model Degree', fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

### 5.3 $R^2$ Comparison

The **coefficient of determination** $R^2$ measures the fraction of total
variance explained by the model:

$$R^2 = 1 - \frac{\text{SS}_{\text{res}}}{\text{SS}_{\text{tot}}}
= 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$$

$R^2 = 1$ means perfect fit; $R^2 = 0$ means the model is no better than
the mean.  Adding more polynomial terms always increases $R^2$, but at the
risk of **overfitting**.

In [ ]:
degrees = [1, 2, 3]
R2_vals = [R2_lin, R2_quad, R2_cub]

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(degrees, R2_vals, color=['steelblue', 'darkorange', 'seagreen'],
              edgecolor='black', linewidth=0.5, width=0.6)
ax.set_xlabel('Polynomial Degree', fontsize=12)
ax.set_ylabel('$R^2$', fontsize=12)
ax.set_title('Coefficient of Determination by Model Degree', fontsize=13)
ax.set_xticks(degrees)
ax.set_ylim(0.9, 1.005)
ax.grid(True, alpha=0.3, axis='y')

for bar, val in zip(bars, R2_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
            f'{val:.5f}', ha='center', va='bottom', fontsize=10)

fig.tight_layout()
plt.show()

---

## 6. Exercises

Test your understanding with the following problems.

### Exercise 1 (Easy) --- Line Fit for 4 Points

Find the least-squares line $y = a + bt$ through the four points

$$(0, 1),\;(1, 3),\;(2, 4),\;(3, 4)$$

**Steps:**
1. Build the $4 \times 2$ Vandermonde matrix.
2. Call `least_squares(V, y)` to get the coefficients.
3. Compute the residual and $\|\mathbf{r}\|^2$.
4. Plot the data and the best-fit line.
5. Verify your coefficients match `np.polyfit(t, y, 1)`.

In [ ]:
# YOUR CODE HERE


### Exercise 2 (Medium) --- Parabola Fit and Residual Comparison

Using the same four points from Exercise 1:

1. Fit a **quadratic** model $y = a + bt + ct^2$ (build a $4 \times 3$ Vandermonde).
2. Compute the residual vector and $\|\mathbf{r}\|^2$ for both the linear and quadratic fits.
3. Verify that $\|\mathbf{r}\|^2_{\text{quad}} \leq \|\mathbf{r}\|^2_{\text{linear}}$.
   Explain **why** this must always be true (think about the column spaces).
4. Plot both fits on the same axes.
5. Confirm that the quadratic residual is orthogonal to all three columns of $V$.

*Hint:* $\text{Col}(V_{\text{linear}}) \subseteq \text{Col}(V_{\text{quad}})$,
so projecting onto a *larger* subspace can only get you *closer* to $\mathbf{b}$.

In [ ]:
# YOUR CODE HERE


### Exercise 3 (Challenge) --- Polynomial Degree Selection with AIC/BIC

Using the temperature data from this notebook, fit polynomial models of
degrees 1 through 5.  For each degree $d$:

1. Build the $(n \times (d+1))$ Vandermonde matrix and solve via `least_squares`.
2. Compute $\text{SS}_{\text{res}}$ and $R^2$.
3. Compute the **Akaike Information Criterion (AIC)** and
   **Bayesian Information Criterion (BIC)**:

$$\text{AIC} = n \ln\!\left(\frac{\text{SS}_{\text{res}}}{n}\right) + 2k$$

$$\text{BIC} = n \ln\!\left(\frac{\text{SS}_{\text{res}}}{n}\right) + k \ln(n)$$

where $n$ is the number of data points and $k = d + 1$ is the number of
fitted parameters.

4. Plot three things:
   - $R^2$ vs. degree (should increase monotonically).
   - AIC and BIC vs. degree (should have a minimum --- the "best" degree).
   - All five fitted curves overlaid on the data.

5. Which degree does AIC select? Which does BIC select? Why might they differ?

*Hint:* AIC and BIC penalize model complexity. $R^2$ alone always favors
more parameters, but AIC/BIC balance fit quality against parsimony.

In [ ]:
# YOUR CODE HERE


---

## Summary

| Concept | Key Takeaway |
|---|---|
| **Overdetermined system** | $m > n$: more equations than unknowns, generically inconsistent |
| **Least-squares solution** | $\hat{\mathbf{x}} = \arg\min \|A\mathbf{x} - \mathbf{b}\|^2$ |
| **Normal equations** | $A^T A \hat{\mathbf{x}} = A^T \mathbf{b}$ --- always has a solution when $\text{rank}(A) = n$ |
| **Residual** | $\mathbf{r} = \mathbf{b} - A\hat{\mathbf{x}}$, orthogonal to $\text{Col}(A)$ |
| **Projection** | $A\hat{\mathbf{x}} = \text{proj}_{\text{Col}(A)} \mathbf{b}$ |
| **Vandermonde matrix** | Column $j$ is $[t_0^j, t_1^j, \ldots, t_{m-1}^j]^T$ |
| **Linear regression** | Degree 1 Vandermonde $\to$ best-fit line |
| **Polynomial regression** | Degree $d$ Vandermonde $\to$ best-fit polynomial of degree $d$ |
| **$R^2$** | $1 - \text{SS}_{\text{res}} / \text{SS}_{\text{tot}}$; fraction of variance explained |
| **Model selection** | AIC/BIC penalize complexity; $R^2$ alone always favors more parameters |

**Next:** In Notebook 5.5 we explore the cross product and its connection to
areas, volumes, and the triple scalar product.